# Member A (Shayan): M1+A0 runs
Run top to bottom on a T4 (Colab or Kaggle). Langfuse keys live in Colab *Secrets* (`LANGFUSE_PK`, `LANGFUSE_SK`), never here. After every run cell, `save()` copies `results/` and `traces/` to Drive `G10/A/`.

In [ ]:
REPO = "https://github.com/<owner>/<repo>.git"   # TODO: set once C has pushed the private repo
!git clone -b a-memory {REPO} repo
%cd repo
!pip install -q -r requirements.txt opentelemetry-sdk opentelemetry-exporter-otlp
import os, base64
from google.colab import userdata, drive
drive.mount("/content/drive")
pk, sk = userdata.get("LANGFUSE_PK"), userdata.get("LANGFUSE_SK")
os.environ["OTEL_EXPORTER_OTLP_ENDPOINT"] = "https://cloud.langfuse.com/api/public/otel"   # US: us.cloud.langfuse.com
os.environ["OTEL_EXPORTER_OTLP_HEADERS"] = "Authorization=Basic%20" + base64.b64encode(f"{pk}:{sk}".encode()).decode()
os.environ.pop("QA_M1_BUDGET", None)   # main runs must use the config budget (768)

def save():
    !mkdir -p /content/drive/MyDrive/G10/A && cp -r results traces /content/drive/MyDrive/G10/A/

# nice to have: C's baseline json, for the efficiency number in the manifest
!mkdir -p results && cp /content/drive/MyDrive/G10/C/results/G10_M0+A0_*_s*.json results/ 2>/dev/null || echo "no baseline yet, running anyway"

In [ ]:
!python tests/test_m1.py

## A2. M1+A0 on dev, 3 seeds (~45-75 min per seed)
If a seed crashes, re-run only it with `--seed 42` instead of `--all-seeds`.

In [ ]:
assert "QA_M1_BUDGET" not in os.environ
!python -m harness.evaluate --group G10 --memory M1 --adapter A0 --split dev --all-seeds
save()

## A3. Memory budget ablation (Step 5 Q2)
Short on GPU? Replace `--all-seeds` with `--seed 13` and say so in `report/parts/ablation.tex`. Tell C: `QA_M1_BUDGET` 256 and 128 under groups `G10ABL256`/`G10ABL128` (for `DEVIATIONS.md`).

In [ ]:
for b in ("256", "128"):
    os.environ["QA_M1_BUDGET"] = b
    !python -m harness.evaluate --group G10ABL{b} --memory M1 --adapter A0 --split dev --all-seeds
    save()
os.environ.pop("QA_M1_BUDGET")
!python -m harness.summarize --group G10 --split dev
!python -m harness.summarize --group G10ABL256 --split dev
!python -m harness.summarize --group G10ABL128 --split dev

## A4. Real traces for MEMORY.md
Prints the `memory_read`/`memory_write` spans of one session (pick sessions from the A2 run). For the correction example, run `python scripts/chat.py --memory M1` in a terminal: "My phone was stolen on 3 March 2025", then "Correction: it was 5 May 2024".

In [ ]:
import glob, json
SESSION = "DEV-T3-0041"
for fp in sorted(glob.glob("traces/G10_M1+A0_dev_s13_*.jsonl")):
    for line in open(fp, encoding="utf-8"):
        if SESSION in line and ('memory_read' in line or 'memory_write' in line):
            print(json.dumps(json.loads(line), indent=1, ensure_ascii=False)[:3000], "
---")

## A5. Secret test (Sat 28 Nov, open 48 h)
Upload `secret_test_questions.jsonl` to the repo root first. **Never commit it.**

In [ ]:
assert "QA_M1_BUDGET" not in os.environ
!python -m harness.evaluate --group G10 --memory M1 --adapter A0 --split test --questions secret_test_questions.jsonl --all-seeds
save()

## A6. Failed turns to label in `errors/ERRORS_A.csv`

In [ ]:
import csv
for seed in (13, 42, 2026):
    for r in csv.DictReader(open(f"results/G10_M1+A0_dev_s{seed}.csv", encoding="utf-8")):
        if r["correct"] == "0" or r["ungrounded_cites"] not in ("", "0"):
            print(r["config"], r["seed"], r["session_id"], r["turn"], r["family"], "ungrounded=" + r["ungrounded_cites"],
                  "abst_exp=" + r["abstain_expected"], "abst=" + r["abstained"], "parse=" + r["parse_ok"], "memrec=" + r["memory_recall"])